# Notebook 00 — LLM Fundamentals & Prompt Engineering

**Week 1 | No toolkit required**

By the end of this notebook you will be able to:
- Explain tokens, context windows, and temperature in plain terms
- Choose a model appropriate for your hardware
- Write effective prompts using roles, few-shot examples, and chain-of-thought
- Apply the multi-model critic pattern to improve output quality
- Make your first raw API call to a local LLM

## Part 1 — Core Concepts

### Tokens

LLMs don't read words — they read *tokens*, roughly 3–4 characters each in English. The sentence *'The quick brown fox'* is about 5 tokens.

Tokens matter for two reasons:
- **Cost:** cloud APIs charge per token
- **Limits:** every model has a maximum context window measured in tokens

### Context Window

The context window is the model's working memory — everything it can 'see' at once: your system prompt, the conversation history, and the current message. When the window fills up, older content is dropped.

Current local models have large context windows:
- `qwen3.5:9b` — 128K tokens
- `qwen3.6:27b` — 128K tokens
- `llama4:scout` — 10M tokens (exceptional)

### Temperature

| Temperature | Behavior | Use for |
|---|---|---|
| 0.0 | Deterministic | Structured output, fact extraction |
| 0.3–0.7 | Balanced | General Q&A, summarization |
| 0.8–1.2 | Creative | Brainstorming, writing |

### Local vs Cloud

| | Local (Ollama) | Cloud (Anthropic, OpenAI) |
|---|---|---|
| **Privacy** | Data stays on your machine | Sent to third party |
| **Cost** | Electricity only | Per-token billing |
| **Setup** | GPU or Colab required | API key only |
| **Quality** | Frontier-competitive (2026) | Still ahead on hardest tasks |
| **Rate limits** | None | Varies by tier |

### Choosing a Model

| VRAM | Model | Ollama command |
|---|---|---|
| None (Colab T4 15 GB) | `qwen3.5:9b` | *(see notebook 01)* |
| 8 GB | `qwen3.5:9b` | `ollama pull qwen3.5:9b` |
| 16 GB | `qwen3.6:35b-a3b` | `ollama pull qwen3.6:35b-a3b` |
| 24 GB | `qwen3.6:27b` | `ollama pull qwen3.6:27b` |

**Dense vs MoE:** `qwen3.6:35b-a3b` is a *mixture-of-experts* model with 35B total parameters but only 3B active per token — fast and memory-efficient. `qwen3.6:27b` is *dense* (all 27B active every token), simpler to deploy, and the strongest open-weight coding model at this scale as of April 2026.

> **Quantization:** Use Q4_K_M variants when available. 4-bit quantization roughly halves VRAM requirements with minimal quality loss.

## Part 2 — Prompt Engineering

How you structure a prompt matters more than which model you use. These patterns apply to any model.

### Pattern 1 — Roles

LLM APIs use three message roles:
- **system** — shapes the model's behavior throughout the conversation
- **user** — what the human says
- **assistant** — what the model says

The system prompt is your most powerful lever. Be specific.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434/v1/chat/completions"
MODEL = "qwen3.5:9b"  # change to match your hardware

def chat(messages, temperature=0.7, model=MODEL):
    """Raw chat call — no toolkit, just requests."""
    r = requests.post(
        OLLAMA_URL,
        json={"model": model, "messages": messages, "temperature": temperature},
        timeout=120
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

vague = chat([
    {"role": "system", "content": "You are helpful."},
    {"role": "user",   "content": "Explain recursion."}
])

specific = chat([
    {"role": "system", "content":
     "You are a Python tutor. Explain concepts with short code examples. "
     "Assume the student knows basic Python but not algorithms."},
    {"role": "user", "content": "Explain recursion."}
])

print("=== VAGUE ===", vague[:400], sep="\n")
print("\n=== SPECIFIC ===", specific[:400], sep="\n")

### Pattern 2 — Few-Shot Examples

Show the model the format you want rather than describing it. It infers the pattern from examples in the conversation history.

In [ ]:
response = chat([
    {"role": "system",    "content": "Classify sentiment as POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user",      "content": "The battery lasts all day."},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user",      "content": "It stopped working after a week."},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user",      "content": "It arrived on time."},
])
print(response)

### Pattern 3 — Chain-of-Thought

For reasoning tasks, ask the model to think step by step *before* answering. This forces decomposition and often catches errors it would otherwise skip.

In [ ]:
direct = chat([
    {"role": "user", "content":
     "Apples cost $0.50, oranges $0.75. I buy 4 apples and 3 oranges. Change from $5?"}
], temperature=0.0)

cot = chat([
    {"role": "user", "content":
     "Apples cost $0.50, oranges $0.75. I buy 4 apples and 3 oranges. Change from $5?\n"
     "Think step by step before giving your final answer."}
], temperature=0.0)

print("Direct:", direct)
print("\nChain-of-thought:", cot)

### Pattern 4 — The Critic Pattern

Asking a model to review its own output helps — but has a fundamental limit: **a model critiquing its own output looks for failures it's structurally unlikely to produce.** If it has a systematic blind spot, self-critique won't catch it. It grades itself on a curve.

**A second model as critic is more reliable.** Different training data, different architecture decisions, different failure modes. It's far more likely to catch what the first model missed.

This scales up to the *advisor pattern* in notebook 09: a fast model generates, a stronger model critiques. You get near-frontier quality at a fraction of the cost.

In [ ]:
# Self-critique (limited — same blind spots)
draft = chat([
    {"role": "system", "content": "You are an expert Python developer."},
    {"role": "user",   "content": "Write a function to find duplicate items in a list."}
])

self_review = chat([
    {"role": "system",    "content": "You are an expert Python developer."},
    {"role": "user",      "content": "Write a function to find duplicate items in a list."},
    {"role": "assistant", "content": draft},
    {"role": "user",      "content": "Review for correctness, edge cases, and performance."}
])

print("Draft:\n", draft)
print("\nSelf-review:\n", self_review)

In [ ]:
# Cross-model critique (recommended)
# Requires two Ollama models. Adjust to match your hardware:
#   24 GB: GENERATOR="qwen3.6:27b",    CRITIC="gemma3:9b"
#   16 GB: GENERATOR="qwen3.6:35b-a3b", CRITIC="qwen3.5:9b"
#   Colab: GENERATOR="qwen3.5:9b",      CRITIC="gemma3:4b"

GENERATOR_MODEL = "qwen3.5:9b"
CRITIC_MODEL    = "gemma3:4b"

draft = chat(
    [{"role": "system", "content": "You are an expert Python developer."},
     {"role": "user",   "content": "Write a function to find duplicate items in a list."}],
    model=GENERATOR_MODEL
)

cross_review = chat(
    [{"role": "system", "content":
      "You are a rigorous code reviewer. Identify bugs, edge cases, and performance issues. "
      "Do not assume the code is correct."},
     {"role": "user", "content": f"Review this function:\n\n{draft}\n\nList specific issues."}],
    model=CRITIC_MODEL
)

print("Generator output:\n", draft)
print("\nCross-model critique:\n", cross_review)

### Pattern 5 — Structured Output

When you need machine-readable output, specify the exact JSON schema and set `temperature=0.0`. In notebook 02 you'll see how `llm_engines` handles validation and retries automatically.

In [ ]:
import json

result = chat([
    {"role": "system",
     "content": "Extract structured data. Respond ONLY with valid JSON. No explanation."},
    {"role": "user",
     "content": "Extract from: 'Invoice from Acme Corp dated 2026-04-01 for $1,250.00'\n"
                  'Return: {"vendor": ..., "date": ..., "amount": ...}'}
], temperature=0.0)

try:
    print(json.loads(result))
except json.JSONDecodeError:
    print("Non-JSON response:", result)

## Exercises

1. **Temperature:** Ask the same creative question at `0.0` and `1.0` three times each. What varies?

2. **System prompts:** Write one vague and one specific system prompt for the same task. Compare quality.

3. **Cross-model critique:** Generate a short essay with one model, critique it with another. Does the critic find anything the generator wouldn't have caught itself?

4. **Structured extraction:** Give the model a paragraph of text and extract a JSON object from it. Verify with `json.loads()`.

---
**Next:** [Notebook 01 — Environment Setup](01_environment_setup.ipynb)